In [1]:
from Model_Building.ModelBuilder import ModelBuilder
import polars as pl
from params.generic_params import DO_NOT_MODEL
from Model_Building.rfe_cv import run_rfe

In [2]:
train = pl.read_parquet(
    r"/Users/johnwragg/Desktop/DS_work/Hastings_Solution/Prepped_Data/train.parquet"
)

In [3]:
test = pl.read_parquet(
    r"/Users/johnwragg/Desktop/DS_work/Hastings_Solution/Prepped_Data/test.parquet"
)

In [4]:
train.head()

Claim_Number,date_of_loss,Notifier,Notification_period,Inception_to_loss,Location_of_incident,Weather_conditions,Vehicle_mobile,Time_hour,Main_driver,PH_considered_TP_at_fault,Incident_details_present,Injury_details_present,TP_type_insd_pass_back,TP_type_driver,TP_type_pass_back,TP_type_pass_front,TP_type_other,TP_type_nk,TP_injury_whiplash,TP_injury_traumatic,TP_injury_fatality,TP_injury_unclear,TP_injury_nk,Incurred,Capped_Incurred,loss_year,loss_month,non_zero_claim_flag,tp_total,tp_passengers,tp_bike_pedest,tp_injured_known,tp_serious_injury_flag,tp_region,tp_region_total
i64,date,cat,i64,i64,cat,cat,cat,i64,cat,cat,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,i32,i8,i32,i64,i64,i64,i64,bool,cat,i64
1532,2008-01-30,"""PH""",0,128,"""Minor Road""","""NORMAL""","""N""",8,"""Other""","""n/k""",1,0,0,0,0,0,0,1,0,0,0,1,0,1536.902642,1536.902642,2008,1,1,1,0,0,0,false,"""northw""",1
6365,2014-10-06,"""Other""",1,13,"""Minor Road""","""NORMAL""","""Y""",16,"""Y""","""N""",1,0,0,1,0,0,0,0,0,0,0,1,1,3010.386467,3010.386467,2014,10,1,1,0,0,0,false,"""none""",0
324,2004-10-13,"""PH""",0,221,"""Minor Road""","""NORMAL""","""Y""",9,"""Other""","""n/k""",1,0,0,0,0,0,0,1,0,0,0,1,1,0.0,0.0,2004,10,0,1,0,0,0,false,"""none""",0
2011,2009-05-16,"""PH""",0,87,"""Minor Road""","""NORMAL""","""N""",2,"""Other""","""n/k""",0,1,0,0,0,0,0,1,0,0,0,1,0,2607.080616,2607.080616,2009,5,1,1,0,0,0,false,"""westmid""",1
2116,2009-08-04,"""PH""",0,271,"""Home Address""","""NORMAL""","""Y""",8,"""Y""","""n/k""",1,0,0,0,0,0,0,1,0,0,0,1,0,718.378093,718.378093,2009,8,1,1,0,0,0,false,"""eastmid""",1


In [5]:
test.head()

Claim_Number,date_of_loss,Notifier,Notification_period,Inception_to_loss,Location_of_incident,Weather_conditions,Vehicle_mobile,Time_hour,Main_driver,PH_considered_TP_at_fault,Incident_details_present,Injury_details_present,TP_type_insd_pass_back,TP_type_driver,TP_type_pass_back,TP_type_pass_front,TP_type_other,TP_type_nk,TP_injury_whiplash,TP_injury_traumatic,TP_injury_fatality,TP_injury_unclear,TP_injury_nk,Incurred,Capped_Incurred,loss_year,loss_month,non_zero_claim_flag,tp_total,tp_passengers,tp_bike_pedest,tp_injured_known,tp_serious_injury_flag,tp_region,tp_region_total
i64,date,cat,i64,i64,cat,cat,cat,i64,cat,cat,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,i32,i8,i32,i64,i64,i64,i64,bool,cat,i64
325,2004-10-14,"""PH""",1,60,"""Main Road""","""WET""","""N""",16,"""Other""","""n/k""",1,0,0,0,0,0,0,2,0,0,0,2,0,2846.297554,2846.297554,2004,10,1,2,0,0,0,false,"""southe""",2
102,2004-01-13,"""PH""",0,205,"""Minor Road""","""NORMAL""","""Y""",9,"""Other""","""n/k""",1,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,2004,1,0,0,0,0,0,false,"""none""",0
7507,2015-05-26,"""Other""",51,48,"""Minor Road""","""WET""","""Y""",8,"""N""","""N""",1,0,0,1,0,0,0,0,0,0,0,1,1,0.0,0.0,2015,5,0,1,0,0,0,false,"""none""",0
7564,2015-06-08,"""Other""",0,30,"""Main Road""","""N/K""","""Y""",7,"""Other""","""N""",1,0,0,1,0,0,0,0,0,0,0,1,0,0.0,0.0,2015,6,0,1,0,0,0,false,"""southe""",1
120,2004-02-05,"""Other""",1,102,"""Main Road""","""NORMAL""","""Y""",20,"""Other""","""n/k""",1,0,0,0,0,0,0,1,0,0,0,1,0,247.14429,247.14429,2004,2,1,1,0,0,0,false,"""eastmid""",1


In [6]:
features = [col for col in train.columns if col not in DO_NOT_MODEL]

In [11]:
mb = ModelBuilder(features, nil_params={}, sev_params={})

In [12]:
mb.fit(train)

In [15]:
len(mb.nil_model.feature_name_)

31

In [16]:
len(mb.sev_model.feature_name_)

31

In [17]:
preds = mb.predict(test)

In [18]:
preds

p_nonnil,severity_capped,expected_capped,expected_cost
f64,f64,f64,f64
0.950673,6831.681357,6494.696707,10037.505724
0.67266,2386.383684,1605.223883,2480.861639
0.705051,1887.866215,1331.042737,2057.116706
0.71103,3561.477054,2532.315469,3913.674827
0.826431,2699.596783,2231.031538,3448.042741
…,…,…,…
0.597885,2061.876398,1232.764521,1905.228449
0.85185,1690.677555,1440.20363,2225.824058
0.515279,3121.362849,1608.372551,2485.727882


In [24]:
pl.concat([test.select("Capped_Incurred"), preds], how="horizontal")

Capped_Incurred,p_nonnil,severity_capped,expected_capped,expected_cost
f64,f64,f64,f64,f64
2846.297554,0.950673,6831.681357,6494.696707,10037.505724
0.0,0.67266,2386.383684,1605.223883,2480.861639
0.0,0.705051,1887.866215,1331.042737,2057.116706
0.0,0.71103,3561.477054,2532.315469,3913.674827
247.14429,0.826431,2699.596783,2231.031538,3448.042741
…,…,…,…,…
618.929826,0.597885,2061.876398,1232.764521,1905.228449
7215.315789,0.85185,1690.677555,1440.20363,2225.824058
0.0,0.515279,3121.362849,1608.372551,2485.727882


In [25]:
# Testing feature selection

In [26]:
train.head()

Claim_Number,date_of_loss,Notifier,Notification_period,Inception_to_loss,Location_of_incident,Weather_conditions,Vehicle_mobile,Time_hour,Main_driver,PH_considered_TP_at_fault,Incident_details_present,Injury_details_present,TP_type_insd_pass_back,TP_type_driver,TP_type_pass_back,TP_type_pass_front,TP_type_other,TP_type_nk,TP_injury_whiplash,TP_injury_traumatic,TP_injury_fatality,TP_injury_unclear,TP_injury_nk,Incurred,Capped_Incurred,loss_year,loss_month,non_zero_claim_flag,tp_total,tp_passengers,tp_bike_pedest,tp_injured_known,tp_serious_injury_flag,tp_region,tp_region_total
i64,date,cat,i64,i64,cat,cat,cat,i64,cat,cat,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,i32,i8,i32,i64,i64,i64,i64,bool,cat,i64
1532,2008-01-30,"""PH""",0,128,"""Minor Road""","""NORMAL""","""N""",8,"""Other""","""n/k""",1,0,0,0,0,0,0,1,0,0,0,1,0,1536.902642,1536.902642,2008,1,1,1,0,0,0,false,"""northw""",1
6365,2014-10-06,"""Other""",1,13,"""Minor Road""","""NORMAL""","""Y""",16,"""Y""","""N""",1,0,0,1,0,0,0,0,0,0,0,1,1,3010.386467,3010.386467,2014,10,1,1,0,0,0,false,"""none""",0
324,2004-10-13,"""PH""",0,221,"""Minor Road""","""NORMAL""","""Y""",9,"""Other""","""n/k""",1,0,0,0,0,0,0,1,0,0,0,1,1,0.0,0.0,2004,10,0,1,0,0,0,false,"""none""",0
2011,2009-05-16,"""PH""",0,87,"""Minor Road""","""NORMAL""","""N""",2,"""Other""","""n/k""",0,1,0,0,0,0,0,1,0,0,0,1,0,2607.080616,2607.080616,2009,5,1,1,0,0,0,false,"""westmid""",1
2116,2009-08-04,"""PH""",0,271,"""Home Address""","""NORMAL""","""Y""",8,"""Y""","""n/k""",1,0,0,0,0,0,0,1,0,0,0,1,0,718.378093,718.378093,2009,8,1,1,0,0,0,false,"""eastmid""",1


In [27]:
X_train_nil = train.select([col for col in train.columns if col not in DO_NOT_MODEL])

In [29]:
target_nil = "non_zero_claim_flag"

In [30]:
y_train_nill = train.select(target_nil)

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold

min_features_to_select = 1  # Minimum number of features to consider
cv = StratifiedKFold(5)
X_train_nil = (
    train.select(features)
    .with_columns(pl.col(pl.Categorical).to_physical().cast(pl.Int32))
    .to_pandas()
)
y_train_nil = train[target_nil].to_numpy()

rfecv = RFECV(
    estimator=LGBMClassifier(verbose=-1, random_state=42),
    step=1,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring="roc_auc",
    min_features_to_select=1,
    n_jobs=2,
)
rfecv.fit(X_train_nil, y_train_nil)

print(f"Optimal number of features: {rfecv.n_features_}")
selected = list(rfecv.get_feature_names_out())

Optimal number of features: 24


In [35]:
sev_target = "Capped_Incurred"

In [43]:
# For severity model
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold

sev_target = "Capped_Incurred"
pos = train.filter(pl.col(target_nil) == 1)

X_train_sev = (
    pos.select(features)
    .with_columns(pl.col(pl.Categorical).to_physical().cast(pl.Int32))
    .to_pandas()
)
y_train_sev = pos[sev_target].to_numpy()

rfecv_sev = RFECV(
    estimator=LGBMRegressor(objective="gamma", verbose=-1, random_state=42),
    step=1,
    cv=KFold(5, shuffle=True, random_state=42),
    scoring="neg_mean_gamma_deviance",
    min_features_to_select=1,
    n_jobs=2,
)
rfecv_sev.fit(X_train_sev, y_train_sev)

print(f"Optimal number of features: {rfecv_sev.n_features_}")
selected_sev = list(rfecv_sev.get_feature_names_out())

Optimal number of features: 26


In [45]:
len(selected_sev)

26

In [46]:
len(selected)

24

In [39]:
import sklearn

In [40]:
sklearn.metrics.get_scorer_names()

['accuracy',
 'adjusted_mutual_info_score',
 'adjusted_rand_score',
 'average_precision',
 'balanced_accuracy',
 'completeness_score',
 'd2_absolute_error_score',
 'd2_brier_score',
 'd2_log_loss_score',
 'explained_variance',
 'f1',
 'f1_macro',
 'f1_micro',
 'f1_samples',
 'f1_weighted',
 'fowlkes_mallows_score',
 'homogeneity_score',
 'jaccard',
 'jaccard_macro',
 'jaccard_micro',
 'jaccard_samples',
 'jaccard_weighted',
 'matthews_corrcoef',
 'mutual_info_score',
 'neg_brier_score',
 'neg_log_loss',
 'neg_max_error',
 'neg_mean_absolute_error',
 'neg_mean_absolute_percentage_error',
 'neg_mean_gamma_deviance',
 'neg_mean_poisson_deviance',
 'neg_mean_squared_error',
 'neg_mean_squared_log_error',
 'neg_median_absolute_error',
 'neg_negative_likelihood_ratio',
 'neg_root_mean_squared_error',
 'neg_root_mean_squared_log_error',
 'normalized_mutual_info_score',
 'positive_likelihood_ratio',
 'precision',
 'precision_macro',
 'precision_micro',
 'precision_samples',
 'precision_weighte

In [8]:
from params.targets import NIL_TARGET

feat_nil = run_rfe(train, features, target=NIL_TARGET, model_type="nil")

Optimal number of features: 24


In [9]:
len(feat_nil)

24

In [10]:
feat_sev = run_rfe(train, features, target="Capped_Incurred", model_type="sev")

Optimal number of features: 26
